# Fine-tune the engine language model on GermanQuAD (GPU)

Starts from the pretrained model of the notebook **pretrain-engine** and trains it to answer questions about a text
(extractive question answering, `engine-finetune`): every question becomes the sequence
`Frage: … / context / Frage: … / Antwort: <answer><eos>`, and only the answer counts in the loss. Long contexts are
cut into windows (windows without the answer learn to answer `<eos>` = "not in this text part").

After every epoch: exact match and F1 on GermanQuAD test questions (a fixed sample of `EVAL_QUESTIONS`, after the
last epoch all of them). The best model by test answer loss is saved as `model/engine_qa.*`.

**Once:** *File → Import Notebook* with this file, *Session options*: **Accelerator GPU T4**, **Internet on**,
*Add Input → Your Work → Notebooks →* **pretrain-engine** (its output contains `model/engine_lm.params`).


In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────
REPO = "https://github.com/RodionKovalenko/rustCrates"
BRANCH = "feature/neural-network-rust"
EPOCHS = 3
LEARNING_RATE = 0.00005  # peak; warmup 100 steps, then cosine decay to 10 %
MICRO_BATCH = 16         # sequences per forward/backward pass (batch 16)
EVAL_QUESTIONS = 300     # test questions for exact match / F1 after each epoch (the last epoch uses all 1839)
PRECISION = "f16"

INPUT_ROOT = "/kaggle/input"
OUTPUT_ROOT = "/kaggle/working"
SOURCE_DIR = "/tmp/rustCrates"


In [ ]:
import glob, json, os, shutil, subprocess, sys, time, zipfile

def run(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env)

if subprocess.run("nvidia-smi", shell=True).returncode != 0:
    raise SystemExit("No GPU: set Session options -> Accelerator -> GPU T4 x2 (Save Version uses the notebook's accelerator), then run again.")

# Rust (the repository's rust-toolchain file selects the version on the first cargo call)
if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Source
if os.path.exists(SOURCE_DIR):
    run(f"git fetch --depth 200 origin {BRANCH} && git checkout -B {BRANCH} FETCH_HEAD", cwd=SOURCE_DIR)
else:
    run(f"git clone --depth 200 --branch {BRANCH} {REPO} {SOURCE_DIR}")
run("git log -1 --oneline", cwd=SOURCE_DIR)


In [ ]:
# cudarc loads the CUDA driver, cuBLAS and NVRTC at run time: make the toolkit libraries findable.
libs = sorted({os.path.dirname(p) for pattern in ("libcublas.so*", "libnvrtc.so*") for p in glob.glob(f"/usr/local/cuda*/**/{pattern}", recursive=True)})
os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("CUDA library dirs:", libs)

# Build (~10 minutes, no CUDA toolkit needed), or reuse the binary of an earlier output when the code is unchanged
features = "--features gpu-engine"
binary = os.path.join(SOURCE_DIR, "target", "release", "wav-transformer")
commit = subprocess.run("git rev-parse HEAD", shell=True, cwd=SOURCE_DIR, capture_output=True, text=True).stdout.strip()
BUILD_INPUTS = "src Cargo.toml Cargo.lock build.rs rust-toolchain"

def reusable_binary():
    for info_path in sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "bin", "build.json"), recursive=True)):
        with open(info_path) as f:
            info = json.load(f)
        cached = os.path.join(os.path.dirname(info_path), "wav-transformer")
        if info.get("features") != features or not os.path.exists(cached):
            continue
        if info.get("commit") == commit or subprocess.run(f"git diff --quiet {info.get('commit')} HEAD -- {BUILD_INPUTS}", shell=True, cwd=SOURCE_DIR).returncode == 0:
            return cached, info
        print(f"Cached binary of {info.get('commit', '?')[:7]} is outdated")
    return None, None

cached, info = reusable_binary()
if cached:
    os.makedirs(os.path.dirname(binary), exist_ok=True)
    shutil.copy(cached, binary)
    os.chmod(binary, 0o755)
    print(f"Using the cached binary built from {info['commit'][:7]}; no build needed")
else:
    run(f"cargo build --release {features}", cwd=SOURCE_DIR)
    info = {"commit": commit, "features": features}

bin_dir = os.path.join(OUTPUT_ROOT, "bin")
os.makedirs(bin_dir, exist_ok=True)
shutil.copy(binary, os.path.join(bin_dir, "wav-transformer"))
with open(os.path.join(bin_dir, "build.json"), "w") as f:
    json.dump(info, f)


In [ ]:
# Pretrained model from the input (output of pretrain-engine)
storage = os.path.join(SOURCE_DIR, "STORAGE")
os.makedirs(storage, exist_ok=True)
found = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "model", "engine_lm.params"), recursive=True))
if not found:
    raise SystemExit("No model/engine_lm.params in the input: Add Input -> Your Work -> Notebooks -> pretrain-engine.")
pretrained = os.path.dirname(found[0])
for name in ("engine_lm.params", "engine_lm.json", "engine_lm.state.json"):
    shutil.copy(os.path.join(pretrained, name), storage)
print("Pretrained model from", pretrained, open(os.path.join(pretrained, "engine_lm.state.json")).read())


In [ ]:
# Fine-tune; the full log is kept in the output as finetune.log
env = dict(os.environ, MICRO_BATCH=str(MICRO_BATCH), EVAL_QUESTIONS=str(EVAL_QUESTIONS))
cmd = [binary, "engine-finetune", "datasets/qa", str(EPOCHS), str(LEARNING_RATE), PRECISION]
print("$", " ".join(cmd), flush=True)
started = time.time()
with open(os.path.join(OUTPUT_ROOT, "finetune.log"), "a") as log:
    proc = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        print(f"[{(time.time() - started) / 3600:5.2f} h] {line.rstrip()}", flush=True)
    proc.wait()
print("exit code", proc.returncode)


In [ ]:
# Output: the fine-tuned model (parameters, config, state, test history)
output = os.path.join(OUTPUT_ROOT, "model")
os.makedirs(output, exist_ok=True)
for name in os.listdir(storage):
    if name.startswith("engine_qa"):
        shutil.copy(os.path.join(storage, name), output)
print("Saved", sorted(os.listdir(output)))
